In [1]:
import torch
import torch.nn as nn
import os
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [2]:
lstm = nn.LSTM(3, 3)  # Input dim is 3, output dim is 3
inputs = [torch.randn(1, 3) for _ in range(5)]  # make a sequence of length 5

# initialize the hidden state.
hidden = (torch.randn(1, 1, 3),
          torch.randn(1, 1, 3))
for i in inputs:
    # Step through the sequence one element at a time.
    # after each step, hidden contains the hidden state.
    out, hidden = lstm(i.view(1, 1, -1), hidden)

# alternatively, we can do the entire sequence all at once.
# the first value returned by LSTM is all of the hidden states throughout
# the sequence. the second is just the most recent hidden state
# (compare the last slice of "out" with "hidden" below, they are the same)
# The reason for this is that:
# "out" will give you access to all hidden states in the sequence
# "hidden" will allow you to continue the sequence and backpropagate,
# by passing it as an argument  to the lstm at a later time
# Add the extra 2nd dimension
inputs = torch.cat(inputs).view(len(inputs), 1, -1)
hidden = (torch.randn(1, 1, 3), torch.randn(1, 1, 3))  # clean out hidden state
out, hidden = lstm(inputs, hidden)
print(out)
print(hidden)

tensor([[[-0.0187,  0.1713, -0.2944]],

        [[-0.3521,  0.1026, -0.2971]],

        [[-0.3191,  0.0781, -0.1957]],

        [[-0.1634,  0.0941, -0.1637]],

        [[-0.3368,  0.0959, -0.0538]]], grad_fn=<MkldnnRnnLayerBackward0>)
(tensor([[[-0.3368,  0.0959, -0.0538]]], grad_fn=<StackBackward0>), tensor([[[-0.9825,  0.4715, -0.0633]]], grad_fn=<StackBackward0>))


In [16]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

with open("emotion_data/train_text.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
f.close()

with open("emotion_data/train_labels.txt", "r") as f:
    labels = f.readlines()

print(len(lines))
label_clean = []
for l in labels :
   label_clean.append( int(l.replace("\n", "")))
print("labels: ", label_clean)


training_data = [
    # Tags are: DET - determiner; NN - noun; V - verb
    # For example, the word "The" is a determiner
    ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]
word_to_ix = {}
# For each words-list (sentence) and tags-list in each tuple of training_data
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:  # word has not been assigned an index yet
            word_to_ix[word] = len(word_to_ix)  # Assign each word with a unique index
print(word_to_ix)
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}  # Assign each tag with a unique index

# These will usually be more like 32 or 64 dimensional.
# We will keep them small, so we can see how the weights change as we train.
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

3257
labels:  [2, 0, 1, 0, 3, 0, 3, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 2, 2, 0, 2, 3, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 3, 3, 1, 2, 1, 3, 0, 0, 0, 0, 0, 3, 3, 1, 1, 0, 1, 0, 1, 1, 2, 0, 0, 1, 3, 0, 1, 0, 3, 0, 3, 0, 0, 1, 0, 0, 3, 0, 1, 0, 3, 1, 0, 3, 3, 2, 3, 3, 0, 0, 0, 1, 0, 0, 1, 0, 0, 3, 3, 1, 0, 0, 1, 0, 2, 0, 3, 0, 0, 3, 0, 0, 0, 3, 3, 1, 3, 3, 1, 2, 2, 3, 0, 3, 2, 0, 1, 0, 0, 1, 3, 0, 0, 1, 2, 3, 1, 3, 0, 3, 0, 1, 3, 1, 2, 3, 3, 0, 3, 3, 0, 2, 3, 0, 3, 1, 1, 1, 2, 0, 0, 0, 1, 3, 0, 0, 3, 3, 3, 1, 0, 0, 0, 0, 1, 3, 1, 3, 0, 0, 0, 1, 3, 2, 1, 0, 1, 0, 3, 0, 1, 1, 1, 1, 1, 2, 1, 0, 0, 2, 1, 0, 0, 2, 1, 3, 1, 0, 3, 0, 1, 3, 3, 1, 3, 0, 3, 1, 3, 1, 2, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 3, 2, 0, 2, 0, 1, 3, 1, 3, 2, 1, 3, 0, 3, 0, 3, 0, 0, 2, 0, 0, 2, 1, 3, 0, 0, 0, 0, 2, 1, 0, 3, 0, 0, 1, 0, 1, 1, 1, 0, 3, 0, 0, 3, 1, 0, 1, 3, 0, 1, 2, 1, 1, 0, 3, 0, 1, 0, 1, 2, 0, 3, 1, 3, 1, 3, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 2, 1, 1, 0, 0, 1, 0, 2, 2, 3, 1, 0, 3, 0, 1, 1, 0, 0

In [4]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, num_classes):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.hidden2label = nn.Linear(hidden_dim, num_classes)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)  # [seq_len, embedding_dim]
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        last_output = lstm_out[-1]               # [1, hidden_dim]
        scores = self.hidden2label(last_output)  # [1, num_classes]
        return F.log_softmax(scores, dim=1)

In [6]:
#model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
model = LSTMClassifier(128, 256, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# See what the scores are before training
# Note that element i,j of the output is the score for tag j for word i.
# Here we don't need to train, so the code is wrapped in torch.no_grad()
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print(tag_scores)

for epoch in range(300):  # again, normally you would NOT do 300 epochs, it is toy data
    for sentence, label in training_data:
        # Step 1. Remember that Pytorch accumulates gradients.
        # We need to clear them out before each instance
        model.zero_grad()

        # Step 2. Get our inputs ready for the network, that is, turn them into
        # Tensors of word indices.
        #sentence_in = prepare_sequence(sentence, word_to_ix)
        #target = prepare_sequence(tags, tag_to_ix)

#torch.tensor(idxs, dtype=torch.long)
        # Step 3. Run our forward pass.
        output = model(sentence)

        # Step 4. Compute the loss, gradients, and update the parameters by
        #  calling optimizer.step()
        loss = loss_function(output, torch.tensor([label]))
        loss.backward()
        optimizer.step()

# See what the scores are after training
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)

    # The sentence is "the dog ate the apple".  i,j corresponds to score for tag j
    # for word i. The predicted tag is the maximum scoring tag.
    # Here, we can see the predicted sequence below is 0 1 2 0 1
    # since 0 is index of the maximum value of row 1,
    # 1 is the index of maximum value of row 2, etc.
    # Which is DET NOUN VERB DET NOUN, the correct sequence!
    print(tag_scores)

tensor([[-1.0573, -1.0567, -1.1874]])


TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not list